# Traing a model using the pre-trained Graph Neural network to predict the patterns.

In [1]:
# Add the path to the system path list
import sys
from pathlib import Path
# Get the current working directory
current_dir = Path.cwd()

# Assuming your notebook is in a subdirectory of the project, adjust as needed
project_root = current_dir.parent.parent

# Add the project root to sys.path
sys.path.append(str(project_root))
import spatialformer as sp

1) building the graph for each gene within each cell

In [2]:
#if you use the simulation data, suggested threshold should be 10
#if the Xenium data provided, suggested threshold can be 3-5

dataloader = sp.pp.build_graph(data_path = "/scratch/project_465001027/Spatialformer/downstream/subcellular_localization_prediction/data/transcripts.csv",
                 vocab_path = "/scratch/project_465001027/Spatialformer/spatialformer/tokenizer/tokenv3.json",
                 batch_size = 32,
                 graph_path = "/scratch/project_465001027/Spatialformer/downstream/subcellular_localization_prediction/data",
                 threshold = 10,
                 split = True)
train_dataloader, val_dataloader, test_dataloader = dataloader


loading the saved data


2) loading the pre-trained GraphSAGE model

In [3]:



#Step 2: Loading the pre-trained model

model = sp.pp.load_pretrained_model(
                            vocab_path = "/scratch/project_465001027/Spatialformer/spatialformer/tokenizer/tokenv3.json",
                            hidden_dim = 256,
                            output_dim = 512,
                            batch_size = 32,
                            checkpoint = "/scratch/project_465001027/Spatialformer/output/GraphSAGE_model/checkpoints/step=0010000-train_loss=0.3983-val_loss=0.0000-train_acc=0.8256.ckpt",
                            device = "cuda")


3) training the pattern model

In [5]:
sp.pp.train_pattern_model(
                        model,
                        lr = 0.0001,
                        output_dim = 512,
                        strategy = "ddp_notebook",
                        output_dir = "/scratch/project_465001027/Spatialformer/output/GraphSAGE_model/checkpoints",
                        train_dataloader = train_dataloader,
                        val_dataloader = val_dataloader)

training the model


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: junwang666 (junwanggroup). Use `wandb login --relogin` to force relogin


/scratch/project_465001027/deeploc_torch/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:166: PossibleUserWarning: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /scratch/project_465001027/deeploc_torch/lib/python3 ...
  rank_zero_warn(
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


RuntimeError: Lightning can't create new processes if CUDA is already initialized. Did you manually call `torch.cuda.*` functions, have moved the model to the device, or allocated memory on the GPU any other way? Please remove any such calls, or change the selected strategy. You will have to restart the Python kernel.

In [ ]:
model.eval()
embeddings = model(self.train_dataset.x, self.train_dataset.edge_index)

In [ ]:
class GraphSAGESpatialEmbedding(nn.Module):
    def __init__(self, freeze, embedding_path):
        super().__init__()
        pretrained_weights = pickle.load(open(embedding_path, "rb"))
        self.emb = nn.Embedding.from_pretrained(pretrained_weights, freeze=freeze)
        print("require grad:", self.emb.weight.requires_grad)

    def forward(self, x):
        return self.emb(x)

In [6]:
import pickle
pretrained_weights = pickle.load(open("/scratch/project_465001027/Spatialformer/spatial_embeddings/gene_embeddings_GraphSAGE_pandavid.pkl", "rb"))

In [8]:
pretrained_weights.shape

torch.Size([1949, 512])